In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# Fetch stock data
def fetch_stock_data(ticker):
    df = yf.download(ticker, period='1y')
    return df[['Close']]

In [ ]:
# Preprocess data
def preprocess_data(data, time_step=60):
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)

    X, y = [], []
    for i in range(time_step, len(data_scaled)):
        X.append(data_scaled[i - time_step:i])
        y.append(data_scaled[i])

    X = np.array(X, dtype=np.float32)   # (N, time_step, 1)
    y = np.array(y, dtype=np.float32)   # (N, 1)
    return X, y, scaler

In [ ]:
# Split data
def split_data(X, y, split_ratio=0.8):
    split = int(len(X) * split_ratio)
    return X[:split], X[split:], y[:split], y[split:]

In [ ]:
# ── PyTorch Models ──────────────────────────────────────────────────────────

class RNNModel(nn.Module):
    """Vanilla RNN → Dropout → Dense → Dropout → Output"""
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2, dense_units=50):
        super().__init__()
        self.rnn     = nn.RNN(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, dense_units)
        self.relu    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2     = nn.Linear(dense_units, 1)

    def forward(self, x):
        _, h = self.rnn(x)          # h: (1, batch, hidden)
        out = self.dropout(h.squeeze(0))
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out)


class LSTMModel(nn.Module):
    """LSTM → Dropout → Dense → Dropout → Output"""
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2, dense_units=50):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, dense_units)
        self.relu    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2     = nn.Linear(dense_units, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        out = self.dropout(h.squeeze(0))
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out)


class GRUModel(nn.Module):
    """GRU → Dropout → Dense → Dropout → Output"""
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2, dense_units=50):
        super().__init__()
        self.gru     = nn.GRU(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, dense_units)
        self.relu    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc2     = nn.Linear(dense_units, 1)

    def forward(self, x):
        _, h = self.gru(x)
        out = self.dropout(h.squeeze(0))
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out)


class CNN1DModel(nn.Module):
    """
    1-D CNN: Conv1d → ReLU → MaxPool → Flatten → Dense → Dropout → Output
    NOTE: PyTorch Conv1d expects (batch, channels, length), so we transpose
    the input inside forward().
    """
    def __init__(self, time_step=60, input_size=1,
                 filters=50, kernel_size=3,
                 dropout=0.2, dense_units=50):
        super().__init__()
        self.conv    = nn.Conv1d(input_size, filters, kernel_size)
        self.relu    = nn.ReLU()
        self.pool    = nn.MaxPool1d(kernel_size=2)
        # Compute flattened size
        conv_len     = (time_step - kernel_size + 1) // 2
        flat_size    = filters * conv_len
        self.fc1     = nn.Linear(flat_size, dense_units)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(dense_units, 1)

    def forward(self, x):
        # x: (batch, time_step, 1) → (batch, 1, time_step)
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv(x)))
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

In [ ]:
MODEL_REGISTRY = {
    'RNN':    RNNModel,
    'LSTM':   LSTMModel,
    'GRU':    GRUModel,
    '1D-CNN': CNN1DModel,
}

def build_model(model_type, time_step, units=50, dropout_rate=0.2, dense_units=50):
    cls = MODEL_REGISTRY[model_type]
    if model_type == '1D-CNN':
        model = cls(time_step=time_step, filters=units,
                    dropout=dropout_rate, dense_units=dense_units)
    else:
        model = cls(hidden_size=units, dropout=dropout_rate, dense_units=dense_units)
    return model.to(DEVICE)

In [ ]:
# Training loop
def train_model(model, X_train, y_train, epochs=10, batch_size=32, lr=1e-3):
    X_t = torch.from_numpy(X_train).to(DEVICE)   # (N, T, 1)
    y_t = torch.from_numpy(y_train).to(DEVICE)   # (N, 1)

    dataset   = TensorDataset(X_t, y_t)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        print(f'  Epoch {epoch}/{epochs}  loss={epoch_loss/len(dataset):.6f}')

In [ ]:
# Inference helper
@torch.no_grad()
def predict(model, X_np):
    model.eval()
    X_t = torch.from_numpy(X_np).to(DEVICE)
    return model(X_t).cpu().numpy()

In [ ]:
# Predict future 7 days
@torch.no_grad()
def predict_future(model, last_sequence, scaler, days=7):
    model.eval()
    predictions  = []
    current_seq  = last_sequence.copy()   # (time_step, 1)  float32

    for _ in range(days):
        x_t  = torch.from_numpy(current_seq[np.newaxis]).to(DEVICE)  # (1, T, 1)
        pred = model(x_t).item()
        predictions.append(pred)
        current_seq = np.append(current_seq[1:], [[pred]], axis=0)

    return scaler.inverse_transform(
        np.array(predictions, dtype=np.float32).reshape(-1, 1)
    ).flatten()

In [ ]:
# Run full pipeline for one model
def run_model_pipeline(model_name, df):
    print(f'\n=== {model_name} ===')
    X, y, scaler = preprocess_data(df)
    time_step    = X.shape[1]
    X_train, X_test, y_train, y_test = split_data(X, y)

    model = build_model(model_name, time_step)
    train_model(model, X_train, y_train)

    y_pred = predict(model, X_test)                                 # (N, 1)
    y_test_inv = scaler.inverse_transform(y_test).flatten()
    y_pred_inv = scaler.inverse_transform(y_pred).flatten()

    mse = mean_squared_error(y_test_inv, y_pred_inv)
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    print(f'  MSE={mse:.4f}  MAE={mae:.4f}')

    last_sequence = X[-1]   # (time_step, 1)  float32
    future = predict_future(model, last_sequence, scaler)

    return {
        'MSE': mse, 'MAE': mae,
        'Future 7 Days': future,
        'actual': y_test_inv,
        'predictions': y_pred_inv,
    }

In [ ]:
# Plot predictions for one model
def plot_predictions(model_name, y_test_inv, y_pred_inv):
    plt.figure(figsize=(10, 6))
    plt.plot(y_test_inv, label='Actual',    color='blue')
    plt.plot(y_pred_inv, label='Predicted', color='red', linestyle='--')
    plt.title(f'{model_name} – Actual vs Predicted Stock Price')
    plt.xlabel('Time')
    plt.ylabel('Stock Price')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Bar chart comparing MSE / MAE across models
def plot_loss_comparison(results):
    model_names = list(results.keys())
    mse_values  = [results[m]['MSE'] for m in model_names]
    mae_values  = [results[m]['MAE'] for m in model_names]

    x     = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width/2, mse_values, width, label='MSE', color='b')
    ax.bar(x + width/2, mae_values, width, label='MAE', color='g')
    ax.set_xlabel('Model')
    ax.set_ylabel('Loss (MSE / MAE)')
    ax.set_title('Model Loss Comparison (MSE vs MAE)')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Main
def main():
    ticker = input('Enter the stock ticker (e.g., AAPL, MSFT, TSLA): ')
    df     = fetch_stock_data(ticker)

    results = {}
    for model_name in ['RNN', 'LSTM', 'GRU', '1D-CNN']:
        results[model_name] = run_model_pipeline(model_name, df)

    plot_loss_comparison(results)

    for model_name in ['RNN', 'LSTM', 'GRU', '1D-CNN']:
        plot_predictions(model_name,
                         results[model_name]['actual'],
                         results[model_name]['predictions'])

    best_mse = min(results, key=lambda m: results[m]['MSE'])
    best_mae = min(results, key=lambda m: results[m]['MAE'])
    print(f'\nBest model by MSE : {best_mse}')
    print(f'Best model by MAE : {best_mae}')
    print('\nFuture 7-day predictions (best MSE model):')
    print(results[best_mse]['Future 7 Days'])


main()